# T0b — mesures GPU (Colab)

Réduction d'incertitude, pas construction (brief §14). Ce notebook mesure, sur GPU :

1. la **synthèse** Chatterbox par scène (§7.2) : temps, débit de parole réel,
   reproductibilité à graine fixée ;
2. le **post-traitement** §7.2 : rognage des bords à −45 dBFS (40 ms gardées),
   passe-haut 80 Hz, gain statique vers −18 LUFS ;
3. l'**alignement mot à mot** par trois candidats (MMS, wav2vec2 FR, Whisper) et
   leur accord à ±33 ms (1 frame, tolérance de R13) ;
4. la survie du **tatouage** Perth à l'AAC 128 kb/s.

**Mode d'emploi**
1. Exécution → Modifier le type d'exécution → **GPU T4** (offre gratuite) ou **L4**.
   La production tourne sur L4 ; une T4 est plus lente, donc ses temps sont un
   **majorant** : si le budget tient sur T4, il tient sur L4. Le reste (alignement,
   débit, post-traitement, tatouage) ne dépend pas de la carte.
2. Exécuter la cellule 1, puis la 2 (installation). La cellule 2 **redémarre la
   session d'elle-même** à la fin (Colab affiche « session plantée » : c'est voulu).
   Reprendre ensuite à la **cellule 3**, sans relancer la 2.
3. Tout exécuter jusqu'au bout. La dernière cellule télécharge `resultats_t0b.zip`.
4. Déposer l'archive dans le dépôt : `mesures/t0b/resultats_t0b.zip`.

**Voix** : par défaut, la voix intégrée de Chatterbox (mesures seulement). Une
voix réelle se lit depuis **Google Drive**, jamais depuis le dépôt, qui est
public (cellule 4).

In [ ]:
# 1. GPU : L4 (celui de la production) ou T4 (offre gratuite, plus lente : temps = majorant).
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert gpu, "Aucun GPU : Exécution → Modifier le type d'exécution → GPU T4 ou L4"
TEMPS_VALENT = "production (L4)" if "L4" in gpu else "majorant : carte plus lente que la L4 de production" if "T4" in gpu else "carte non prévue : temps non comparables"
print("temps mesurés :", TEMPS_VALENT)

In [ ]:
# 2. Versions ÉPINGLÉES. chatterbox-tts 0.1.7 impose torch 2.6.0. Le torchvision
#    préinstallé par Colab, compilé pour un torch plus récent, casse transformers
#    (« torchvision::nms does not exist », puis « _HAS_OPS » même en 0.21.0) : on le
#    RETIRE. Rien ici n'en a besoin, et transformers ne l'importe que s'il existe.
#    La session REDÉMARRE ensuite d'elle-même (message « session plantée » : normal),
#    pour que les nouvelles versions soient chargées. Reprendre à la cellule 3.
!pip uninstall -y -q torchvision
!pip install -q chatterbox-tts==0.1.7 torch==2.6.0 torchaudio==2.6.0 openai-whisper==20250625 resemble-perth==1.0.1 pyloudnorm
!pip uninstall -y -q torchvision
import os
os.kill(os.getpid(), 9)

In [ ]:
# 3. Dépôt, poids épinglés, environnement relevé.
import os, json, hashlib, time, re, unicodedata, difflib, shutil, subprocess, platform
import numpy as np, torch, torchaudio
from huggingface_hub import snapshot_download

DEPOT = "https://github.com/Johnfrancis6/Hypermaker"
if not os.path.exists("Hypermaker"):
    subprocess.run(["git", "clone", "-q", DEPOT], check=True)
COMMIT = subprocess.run(["git", "-C", "Hypermaker", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
SCRIPTS = json.load(open("Hypermaker/t0b/scripts.json", encoding="utf-8"))

CHATTERBOX_REPO, CHATTERBOX_REV = "ResembleAI/chatterbox", "5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18"   # MIT
W2V_REPO, W2V_REV = "jonatasgrosman/wav2vec2-large-xlsr-53-french", "7c79e105a6525d38e1e69f640b974b4a679723cc"  # Apache-2.0
WHISPER_MODELE = "large-v3"

poids = snapshot_download(CHATTERBOX_REPO, revision=CHATTERBOX_REV, allow_patterns=[
    "ve.pt", "t3_mtl23ls_v2.safetensors", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"])

from importlib.metadata import version
import importlib.util
VERSIONS = {"torch": torch.__version__, "torchaudio": torchaudio.__version__}
assert VERSIONS["torch"].startswith("2.6.0") and VERSIONS["torchaudio"].startswith("2.6.0"), \
    f"versions chargées {VERSIONS} : la session n'a pas redémarré après la cellule 2 (Exécution → Redémarrer la session), puis reprendre ici"
assert importlib.util.find_spec("torchvision") is None, "torchvision encore installé : relancer la cellule 2"
ENV = {
    "commit_depot": COMMIT, "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "torchaudio": torchaudio.__version__, "torchvision": "retiré", "chatterbox_tts": version("chatterbox-tts"),
    "openai_whisper": version("openai-whisper"), "resemble_perth": version("resemble-perth"),
    "temps_valent_pour": TEMPS_VALENT, "chatterbox_rev": CHATTERBOX_REV, "wav2vec2_rev": W2V_REV, "whisper": WHISPER_MODELE,
    "ffmpeg": subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0],
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2, ensure_ascii=False))
assert torch.cuda.is_available()
OUT = "resultats_t0b"
shutil.rmtree(OUT, ignore_errors=True)
for d in ("brut", "post", "alignements"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

In [ ]:
# 4. Modèle et voix. VOIX = None : voix intégrée de Chatterbox (mesures seulement).
#    Pour une voix réelle : la déposer dans Google Drive, puis
#       from google.colab import drive; drive.mount('/content/drive')
#       VOIX = '/content/drive/MyDrive/<chemin>.wav'
#    JAMAIS un fichier du dépôt Hypermaker, qui est public.
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
VOIX = None
assert VOIX is None or not os.path.abspath(VOIX).startswith(os.path.abspath("Hypermaker")), "voix dans le dépôt public : refusé"
tts = ChatterboxMultilingualTTS.from_local(poids, "cuda")
if VOIX:
    tts.prepare_conditionals(VOIX)
ENV["voix"] = "intégrée Chatterbox (conds.pt)" if VOIX is None else "fichier Drive (chemin non consigné)"
ENV["sr_synthese"] = tts.sr
print("fréquence de synthèse :", tts.sr)

In [ ]:
# 5. Synthèse : UNE passe par scène, graine = sha256(projet_id + scene_id) mod 2^31 (§7.2).
#    Chaque scène est synthétisée DEUX fois à graine égale : le fichier est-il identique ?
#    Une synthèse d'échauffement, hors mesure, absorbe le coût du premier appel.
import random
def graine(scene_id):
    return int(hashlib.sha256((SCRIPTS["projet_id"] + scene_id).encode()).hexdigest(), 16) % 2**31

def synthese(texte, g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)
    torch.cuda.synchronize(); t = time.perf_counter()
    wav = tts.generate(texte, language_id=SCRIPTS["langue"])
    torch.cuda.synchronize()
    return wav, time.perf_counter() - t

t_froid = synthese("Échauffement du modèle.", 1)[1]
MESURES = {"echauffement_s": round(t_froid, 3), "scenes": {}}
for sc in SCRIPTS["scenes"]:
    g = graine(sc["scene_id"])
    fichiers, temps = [], []
    for essai in (1, 2):
        wav, dt = synthese(sc["texte"], g)
        f = f"{OUT}/brut/{sc['scene_id']}_essai{essai}.wav"
        torchaudio.save(f, wav.cpu(), tts.sr)
        fichiers.append(f); temps.append(dt)
    h = [hashlib.sha256(np.ascontiguousarray(torchaudio.load(f)[0].numpy()).tobytes()).hexdigest() for f in fichiers]
    duree = torchaudio.info(fichiers[0]).num_frames / tts.sr
    MESURES["scenes"][sc["scene_id"]] = {
        "texte": sc["texte"], "mots": len(sc["texte"].split()), "graine": g,
        "synthese_s": [round(x, 3) for x in temps], "duree_brute_s": round(duree, 3),
        "facteur_temps_reel": round(temps[0] / duree, 3),
        "echantillons_identiques_entre_essais": h[0] == h[1],
    }
    print(sc["scene_id"], MESURES["scenes"][sc["scene_id"]])
tot = sum(m["synthese_s"][0] for m in MESURES["scenes"].values())
MESURES["synthese_totale_8_scenes_s"] = round(tot, 3)
print("synthèse des 8 scènes (essai 1) :", round(tot, 1), "s")

In [ ]:
# 6. Post-traitement §7.2, identique pour toutes les scènes (essai 1) :
#    rognage des bords sous -45 dBFS en gardant 40 ms, passe-haut 80 Hz, 48 kHz,
#    puis GAIN STATIQUE mesuré vers -18 LUFS (pas de compresseur : il dépend du contenu).
import pyloudnorm as pyln
ROGNAGE = ("silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.04,"
           "areverse,silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.04,areverse")
def ff(*args):
    r = subprocess.run(["ffmpeg", "-v", "error", "-y", *args], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
for sid, m in MESURES["scenes"].items():
    tmp = f"{OUT}/post/{sid}_avant_gain.wav"
    ff("-i", f"{OUT}/brut/{sid}_essai1.wav", "-af", f"{ROGNAGE},highpass=f=80,aresample=48000", "-ac", "1", tmp)
    x, sr = torchaudio.load(tmp)
    lufs = pyln.Meter(sr).integrated_loudness(x[0].numpy())
    gain = -18.0 - lufs
    ff("-i", tmp, "-af", f"volume={gain:.4f}dB", "-c:a", "pcm_s24le", f"{OUT}/post/{sid}.wav")
    os.remove(tmp)
    y, _ = torchaudio.load(f"{OUT}/post/{sid}.wav")
    m.update({"lufs_avant_gain": round(lufs, 2), "gain_statique_db": round(gain, 2),
              "lufs_apres": round(pyln.Meter(48000).integrated_loudness(y[0].numpy()), 2),
              "duree_post_s": round(y.shape[1] / 48000, 3),
              "crete_dbfs": round(20 * np.log10(float(y.abs().max()) + 1e-12), 2)})
    print(sid, {k: m[k] for k in ("duree_brute_s", "duree_post_s", "lufs_avant_gain", "gain_statique_db", "lufs_apres", "crete_dbfs")})

In [ ]:
# 7. Alignement mot à mot, trois candidats, sur l'audio POST-TRAITÉ (celui qui sera monté).
#    Temps en secondes LOCALES à la scène (§7.3). Mots du script : séparés par les espaces,
#    ponctuation de bord retirée ; « recevez-le », « l'Afrique » restent un mot.
import whisper
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from torchaudio.functional import forced_align, merge_tokens

def mots_script(texte):
    return [w.strip(".,;:!?«»\"") for w in texte.split()]
def sans_accents(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
def cle(w):
    return re.sub(r"[^a-z0-9]", "", sans_accents(w.lower().replace("’", "'")))
def audio16k(sid):
    x, sr = torchaudio.load(f"{OUT}/post/{sid}.wav")
    return torchaudio.functional.resample(x, sr, 16000)

# a) MMS_FA (torchaudio) : alphabet a-z et apostrophe ; accents retirés, tiret retiré.
mms = torchaudio.pipelines.MMS_FA
mms_modele = mms.get_model(with_star=False).to("cuda").eval()
mms_tok, mms_ali = mms.get_tokenizer(), mms.get_aligner()
def aligner_mms(sid, mots):
    x = audio16k(sid)
    norm = [re.sub(r"[^a-z']", "", sans_accents(w.lower().replace("’", "'"))) for w in mots]
    with torch.inference_mode():
        em, _ = mms_modele(x.to("cuda"))
    spans = mms_ali(em[0], mms_tok(norm))
    r = x.shape[1] / em.shape[1] / 16000
    return [{"mot": w, "debut_s": round(s[0].start * r, 4), "fin_s": round(s[-1].end * r, 4),
             "confiance": round(float(np.mean([t.score for t in s])), 4)} for w, s in zip(mots, spans)]

# b) wav2vec2 français + alignement forcé CTC ; caractères hors vocabulaire retirés (consignés).
w2v_proc = Wav2Vec2Processor.from_pretrained(W2V_REPO, revision=W2V_REV)
w2v = Wav2Vec2ForCTC.from_pretrained(W2V_REPO, revision=W2V_REV).to("cuda").eval()
vocab = w2v_proc.tokenizer.get_vocab()
DELIM = vocab[w2v_proc.tokenizer.word_delimiter_token]
HORS_VOCAB = set()
def aligner_w2v(sid, mots):
    x = audio16k(sid)
    ids_mots = []
    for w in mots:
        ids = []
        for c in w.lower().replace("’", "'"):
            if c in vocab: ids.append(vocab[c])
            else: HORS_VOCAB.add(c)
        ids_mots.append(ids)
    cibles = []
    for i, ids in enumerate(ids_mots):
        cibles += ids + ([DELIM] if i < len(ids_mots) - 1 else [])
    with torch.inference_mode():
        lp = torch.log_softmax(w2v(w2v_proc(x[0].numpy(), sampling_rate=16000, return_tensors="pt").input_values.to("cuda")).logits, -1).cpu()
    ali, sc = forced_align(lp, torch.tensor([cibles], dtype=torch.int32), blank=w2v_proc.tokenizer.pad_token_id)
    spans = merge_tokens(ali[0], sc[0].exp())
    assert len(spans) == len(cibles)
    r = x.shape[1] / lp.shape[1] / 16000
    out, k = [], 0
    for w, ids in zip(mots, ids_mots):
        s = spans[k:k + len(ids)]
        k += len(ids) + 1
        out.append({"mot": w, "debut_s": round(s[0].start * r, 4), "fin_s": round(s[-1].end * r, 4),
                    "confiance": round(float(np.mean([t.score for t in s])), 4)})
    return out

# c) Whisper : reconnaissance LIBRE, pas un alignement forcé. Ses mots sont rapprochés
#    de ceux du script ; un mot du script non reconnu reste sans horodatage.
wh = whisper.load_model(WHISPER_MODELE, device="cuda")
def aligner_whisper(sid, mots):
    res = wh.transcribe(f"{OUT}/post/{sid}.wav", language="fr", word_timestamps=True, temperature=0)
    rec = [w for seg in res["segments"] for w in seg["words"]]
    a, b = [cle(w) for w in mots], [cle(w["word"]) for w in rec]
    out = [{"mot": w, "debut_s": None, "fin_s": None, "confiance": None} for w in mots]
    for bloc in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_matching_blocks():
        for i in range(bloc.size):
            w = rec[bloc.b + i]
            out[bloc.a + i].update({"debut_s": round(w["start"], 4), "fin_s": round(w["end"], 4), "confiance": round(w["probability"], 4)})
    return out, res["text"].strip()

ALIGN = {}
for sc in SCRIPTS["scenes"]:
    sid, mots = sc["scene_id"], mots_script(sc["texte"])
    t = time.perf_counter(); a_mms = aligner_mms(sid, mots); d_mms = time.perf_counter() - t
    t = time.perf_counter(); a_w2v = aligner_w2v(sid, mots); d_w2v = time.perf_counter() - t
    t = time.perf_counter(); a_wh, transcrit = aligner_whisper(sid, mots); d_wh = time.perf_counter() - t
    ALIGN[sid] = {"mms": a_mms, "wav2vec2_fr": a_w2v, "whisper": a_wh, "transcription_whisper": transcrit,
                  "temps_s": {"mms": round(d_mms, 3), "wav2vec2_fr": round(d_w2v, 3), "whisper": round(d_wh, 3)}}
    json.dump(ALIGN[sid], open(f"{OUT}/alignements/{sid}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(sid, "whisper a entendu :", transcrit)
MESURES["caracteres_hors_vocabulaire_wav2vec2"] = sorted(HORS_VOCAB)

In [ ]:
# 8. Accord entre aligneurs, au début de mot, à ±1 frame (33 ms, R13).
#    Concordant = les trois débuts à moins de 33 ms deux à deux. Le reste est à annoter à la main.
TOL = 1 / 30
a_annoter, n_mots, concord, paires = [], 0, 0, {"mms~wav2vec2_fr": [], "mms~whisper": [], "wav2vec2_fr~whisper": []}
for sid, A in ALIGN.items():
    for i, w in enumerate(A["mms"]):
        n_mots += 1
        d = {k: A[k][i]["debut_s"] for k in ("mms", "wav2vec2_fr", "whisper")}
        for p in paires:
            x, y = p.split("~")
            if d[x] is not None and d[y] is not None:
                paires[p].append(abs(d[x] - d[y]))
        ok = None not in d.values() and max(d.values()) - min(d.values()) <= TOL
        concord += ok
        if not ok:
            a_annoter.append({"scene_id": sid, "index": i, "mot": w["mot"], "debuts_s": d})
MESURES["accord"] = {
    "mots": n_mots, "concordants_33ms": concord, "a_annoter": len(a_annoter),
    "ecarts_ms": {p: {"median": round(1000 * float(np.median(v)), 1), "p95": round(1000 * float(np.percentile(v, 95)), 1),
                      "max": round(1000 * float(np.max(v)), 1), "n": len(v)} for p, v in paires.items() if v},
    "degrade_mms": {sid: round(float(np.mean([w["confiance"] < 0.5 for w in A["mms"]])), 3) for sid, A in ALIGN.items()},
}
json.dump(a_annoter, open(f"{OUT}/mots_a_annoter.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(MESURES["accord"], indent=2, ensure_ascii=False))

# Débit de parole réel (S5 suppose 2,3 mots/s) : mots / (fin du dernier mot - début du premier), selon wav2vec2.
for sid, A in ALIGN.items():
    w = A["wav2vec2_fr"]
    MESURES["scenes"][sid]["debit_mots_par_s"] = round(len(w) / (w[-1]["fin_s"] - w[0]["debut_s"]), 3)
debits = [m["debit_mots_par_s"] for m in MESURES["scenes"].values()]
MESURES["debit_mots_par_s"] = {"moyen": round(float(np.mean(debits)), 3), "min": min(debits), "max": max(debits), "hypothese_S5": 2.3}
print("débit :", MESURES["debit_mots_par_s"])

In [ ]:
# 9. Tatouage Perth : présent sur la synthèse brute, après post-traitement, et après AAC 128 kb/s ?
import perth
wm = perth.PerthImplicitWatermarker()
def score(f):
    # get_watermark renvoie une prédiction par trame (arrondie à 0/1 par défaut) :
    # score = moyenne des prédictions NON arrondies, 0 = absent, 1 = présent.
    x, sr = torchaudio.load(f)
    return round(float(np.mean(wm.get_watermark(x[0].numpy(), sample_rate=sr, round=False))), 4)
TATOUAGE = {}
for sid in MESURES["scenes"]:
    ff("-i", f"{OUT}/post/{sid}.wav", "-c:a", "aac", "-b:a", "128k", f"{OUT}/post/{sid}.m4a")
    ff("-i", f"{OUT}/post/{sid}.m4a", f"{OUT}/post/{sid}_aac.wav")
    TATOUAGE[sid] = {"brut": score(f"{OUT}/brut/{sid}_essai1.wav"), "post": score(f"{OUT}/post/{sid}.wav"),
                     "apres_aac_128k": score(f"{OUT}/post/{sid}_aac.wav")}
    os.remove(f"{OUT}/post/{sid}.m4a"); os.remove(f"{OUT}/post/{sid}_aac.wav")
# Témoin : un audio jamais tatoué doit donner un score bas, sinon le détecteur ne prouve rien.
ff("-f", "lavfi", "-i", "anoisesrc=d=3:c=pink:a=0.1:r=48000", f"{OUT}/temoin_bruit.wav")
TATOUAGE["temoin_non_tatoue"] = score(f"{OUT}/temoin_bruit.wav")
os.remove(f"{OUT}/temoin_bruit.wav")
MESURES["tatouage"] = TATOUAGE
print(json.dumps(TATOUAGE, indent=2))

In [ ]:
# 10. Archive des résultats, à déposer dans le dépôt : mesures/t0b/resultats_t0b.zip
json.dump(ENV, open(f"{OUT}/environnement.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(MESURES, open(f"{OUT}/mesures.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
shutil.make_archive("resultats_t0b", "zip", OUT)
from google.colab import files
files.download("resultats_t0b.zip")